# Supertargeting Demo

This notebook demonstrates a minimal workflow for reading heat-integration stream data from Excel and running automatic supertargeting calculations.

## What This Demo Covers

1. Read the local Excel stream sheet
2. Convert rows into `ThermalStream` objects
3. Run supertargeting for a selected $\Delta T_{\min}$
4. Inspect the stream table, summary metrics, area intervals, composite curve, and balanced composite curve

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

from IPython.display import Image, display

PROCESS_ONLY_UTILITY_NAMES = {'Hot Oil', 'HP Steam', 'Cooling Water', 'Refrigerant 2', 'Fired Heat (1000)'}

NOTEBOOK_DIR = Path.cwd().resolve()
SUPER_RIGHT_DIR = NOTEBOOK_DIR.parent
ROOT = SUPER_RIGHT_DIR.parent
SUPERTARGETING_ROOT = ROOT / '06_HYSYS_Supertargeting'

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(SUPERTARGETING_ROOT) not in sys.path:
    sys.path.insert(0, str(SUPERTARGETING_ROOT))

from tex_tables.generate_supertargeting_area_target import parse_xlsx_rows, stream_rows_from_workbook
from hysys_utils.supertargeting import _rows_to_streams, curve_plot_records, run_supertargeting


In [ ]:
INPUT_XLSX = NOTEBOOK_DIR / 'Input Sheet of 20.xlsx'
IMAGE_DIR = NOTEBOOK_DIR / 'images'
DELTA_TMIN_C = 20.0

INPUT_XLSX

In [ ]:
workbook_rows = parse_xlsx_rows(INPUT_XLSX)
stream_rows = stream_rows_from_workbook(workbook_rows)

streams_df = pd.DataFrame(
    stream_rows,
    columns=['Stream Information', 'Supply Temperature (C)', 'Target Temperature (C)', 'Heat Load (kW)', 'U (kW/m2-K)', 'CP (kW/K)'],
)
streams_df.index = streams_df.index + 1
streams_df

In [ ]:
streams = _rows_to_streams(workbook_rows)
result = run_supertargeting(streams, delta_tmin_c=DELTA_TMIN_C)

summary_df = pd.DataFrame(
    [
        {'Metric': 'Delta Tmin (C)', 'Value': DELTA_TMIN_C},
        {'Metric': 'Pinch hot temperature (C)', 'Value': result.pinch_hot_c},
        {'Metric': 'Pinch cold temperature (C)', 'Value': result.pinch_cold_c},
        {'Metric': 'Minimum hot utility (kW)', 'Value': result.minimum_hot_utility_kw},
        {'Metric': 'Minimum cold utility (kW)', 'Value': result.minimum_cold_utility_kw},
        {'Metric': 'Minimum area (m2)', 'Value': result.minimum_area_m2},
    ]
)
summary_df

## Total Results Table

This consolidated table summarizes the current `\Delta T_{\min}=20^\circ C` case for both all-stream and process-only views.

In [ ]:
process_streams = [stream for stream in streams if stream.name not in PROCESS_ONLY_UTILITY_NAMES]
assert not any(stream.name in PROCESS_ONLY_UTILITY_NAMES for stream in process_streams)
process_result = run_supertargeting(process_streams, delta_tmin_c=DELTA_TMIN_C)

total_results_df = pd.DataFrame(
    [
        {'Basis': 'All streams', 'Metric': 'Stream count', 'Value': len(streams)},
        {'Basis': 'All streams', 'Metric': 'Pinch hot temperature (C)', 'Value': result.pinch_hot_c},
        {'Basis': 'All streams', 'Metric': 'Pinch cold temperature (C)', 'Value': result.pinch_cold_c},
        {'Basis': 'All streams', 'Metric': 'Minimum hot utility (kW)', 'Value': result.minimum_hot_utility_kw},
        {'Basis': 'All streams', 'Metric': 'Minimum cold utility (kW)', 'Value': result.minimum_cold_utility_kw},
        {'Basis': 'All streams', 'Metric': 'Minimum area (m2)', 'Value': result.minimum_area_m2},
        {'Basis': 'Process only', 'Metric': 'Stream count', 'Value': len(process_streams)},
        {'Basis': 'Process only', 'Metric': 'Pinch hot temperature (C)', 'Value': process_result.pinch_hot_c},
        {'Basis': 'Process only', 'Metric': 'Pinch cold temperature (C)', 'Value': process_result.pinch_cold_c},
        {'Basis': 'Process only', 'Metric': 'Minimum hot utility (kW)', 'Value': process_result.minimum_hot_utility_kw},
        {'Basis': 'Process only', 'Metric': 'Minimum cold utility (kW)', 'Value': process_result.minimum_cold_utility_kw},
        {'Basis': 'Process only', 'Metric': 'Minimum area (m2)', 'Value': process_result.minimum_area_m2},
    ]
)
total_results_df

In [ ]:
intervals_df = pd.DataFrame(
    [
        {
            'Delta H Interval (kW)': f'{interval.enthalpy_start_kw:.1f}--{interval.enthalpy_end_kw:.1f}',
            'Th_out (C)': interval.hot_out_c,
            'Th_in (C)': interval.hot_in_c,
            'Tc_in (C)': interval.cold_in_c,
            'Tc_out (C)': interval.cold_out_c,
            'Hot Streams': list(interval.hot_stream_ids),
            'Cold Streams': list(interval.cold_stream_ids),
            'LMTD (C)': interval.log_mean_temperature_difference_c,
            'Area (m2)': interval.area_m2,
        }
        for interval in result.area_intervals
    ]
)
intervals_df

In [ ]:
records_bcc = pd.DataFrame(curve_plot_records(streams, result))
records_cc = pd.DataFrame(curve_plot_records(process_streams, process_result))

pd.concat(
    [
        records_cc.assign(source='process_only_cc'),
        records_bcc.assign(source='all_streams_bcc'),
    ],
    ignore_index=True,
).head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)

plot_specs = [
    (records_cc, 'composite_curve', 'Composite Curve (process streams only)'),
    (records_bcc, 'bcc_curve', 'Balanced Composite Curve (including utility streams)'),
]

for ax, (records_plot, plot_name, title) in zip(axes, plot_specs):
    hot_rows = records_plot[(records_plot['plot_name'] == plot_name) & (records_plot['curve_name'] == 'hot_curve')]
    cold_rows = records_plot[(records_plot['plot_name'] == plot_name) & (records_plot['curve_name'] == 'cold_curve')]
    ax.plot(hot_rows['enthalpy_kw'], hot_rows['temperature_c'], color='#c62828', linewidth=2.2, label='Hot Composite Curve')
    ax.plot(cold_rows['enthalpy_kw'], cold_rows['temperature_c'], color='#1565c0', linewidth=2.2, label='Cold Composite Curve')
    ax.set_title(title)
    ax.set_xlabel('Enthalpy (kW)')
    ax.set_ylabel('Temperature (C)')
    ax.grid(True, color='#d9d9d9', linewidth=0.8)
    ax.legend(frameon=True)

plt.show()


## Rendered Figures

The notebook also shows the pre-rendered figures copied into this `github` demo folder.

In [ ]:
figure_map = {
    'Balanced composite curve': IMAGE_DIR / 'bcc_all_streams_dtmin20.png',
    'Partitioned BCC': IMAGE_DIR / 'bcc_all_streams_partitioned_dtmin20.png',
    'Composite curve without utility streams': IMAGE_DIR / 'composite_curve_process_only_dtmin20.png',
    'ACC / TOC / TAC vs Delta Tmin': IMAGE_DIR / 'dtmin_economics_aligned_bcc.png',
}

pd.DataFrame({'Figure': list(figure_map.keys()), 'Path': [str(path) for path in figure_map.values()]})

In [ ]:
for title, path in figure_map.items():
    print(title)
    display(Image(filename=str(path)))


## Notes

- The demo uses the local copy of `Input Sheet of 20.xlsx` in this folder.
- Change `DELTA_TMIN_C` and rerun the notebook cells to test another case.
- The same backend can be reused later for publishing a cleaner standalone GitHub example.
- The `images/` subfolder is intended to keep the demo notebook self-contained.

## Acknowledgement

The authors also gratefully acknowledge A/Prof. Sachin V. Jangam for providing the HDA Aspen HYSYS case study through the course `CN4205R`, and for his teaching in `CN4205 Pinch Analysis and Process Integration`, from which this work benefited.

## Contact

Questions, feedback, or collaboration ideas are welcome. Please contact: `ziyun_zhang@u.nus.edu`